# 02 - Data Cleaning

**Tasks**:
- Standardize column names
- Convert data types
- Handle missing values
- Remove duplicates
- Validate data quality

**Output**: Cleaned data saved to `data/processed/`

In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)

# S3 paths
BASE = "https://validex-ml-data.s3.us-east-1.amazonaws.com"

# Local output paths
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

In [19]:
# Load raw data
daily = pd.read_parquet(f"{BASE}/daily_adjusted/ALL_daily_adjusted.parquet")
income = pd.read_parquet(f"{BASE}/fundamentals/ALL_income_statement.parquet")
balance = pd.read_parquet(f"{BASE}/fundamentals/ALL_balance_sheet.parquet")
cashflow = pd.read_parquet(f"{BASE}/fundamentals/ALL_cash_flow.parquet")
options = pd.read_parquet(f"{BASE}/options/ALL_options.parquet")
overview = pd.read_csv(f"{BASE}/fundamentals/ALL_overview.csv")

print("Data loaded successfully!")

Data loaded successfully!


## 1. Clean Daily Prices

In [20]:
print(f"Shape: {daily.shape}")
print(f"\nColumns: {daily.columns.tolist()}")
print(f"\nData types:\n{daily.dtypes}")
print(f"\nMissing values:\n{daily.isnull().sum()}")
print(f"\nSample:")
daily.head()

Shape: (52486, 10)

Columns: ['date', 'symbol', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'dividend', 'split_coeff']

Data types:
date           datetime64[ns]
symbol                 object
open                  float64
high                  float64
low                   float64
close                 float64
adj_close             float64
volume                  int64
dividend              float64
split_coeff           float64
dtype: object

Missing values:
date           0
symbol         0
open           0
high           0
low            0
close          0
adj_close      0
volume         0
dividend       0
split_coeff    0
dtype: int64

Sample:


,date,symbol,open,high,low,close,adj_close,volume,dividend,split_coeff
0,2009-08-06,AVGO,16.50,16.91,15.56,16.18,1.155956,24197800,0.0,1.0
1,2009-08-07,AVGO,16.15,16.76,16.03,16.43,1.173816,2454300,0.0,1.0
2,2009-08-10,AVGO,16.63,16.63,15.61,15.97,1.140952,2421000,0.0,1.0
3,2009-08-11,AVGO,15.98,16.00,15.50,15.67,1.119519,2305400,0.0,1.0
4,2009-08-12,AVGO,16.15,16.20,15.66,16.00,1.143096,1451300,0.0,1.0


In [21]:
def clean_daily(df):
    df = df.copy()
    
    # Standardize column names
    df.columns = df.columns.str.lower().str.strip()
    
    # Rename for consistency
    df = df.rename(columns={
        'adj_close': 'adjusted_close',
        'split_coeff': 'split_coefficient'
    })
    
    # Convert date
    df['date'] = pd.to_datetime(df['date'])
    
    # Ensure numeric columns
    numeric_cols = ['open', 'high', 'low', 'close', 'adjusted_close', 'volume', 'dividend', 'split_coefficient']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['symbol', 'date'])
    
    # Sort
    df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
    
    return df

daily_clean = clean_daily(daily)
print(f"Cleaned shape: {daily_clean.shape}")
print(f"Date range: {daily_clean['date'].min()} to {daily_clean['date'].max()}")
print(f"Tickers: {daily_clean['symbol'].unique()}")

Cleaned shape: (52486, 10)
Date range: 2000-01-03 00:00:00 to 2025-12-31 00:00:00
Tickers: ['AAPL' 'AMZN' 'AVGO' 'GOOG' 'GOOGL' 'META' 'MSFT' 'NVDA' 'TSLA' 'WMT']


In [22]:
# Check for missing values after cleaning
print("Missing values:")
print(daily_clean.isnull().sum())

# Check rows per ticker
print("\nRows per ticker:")
print(daily_clean.groupby('symbol').size())

Missing values:
date                 0
symbol               0
open                 0
high                 0
low                  0
close                0
adjusted_close       0
volume               0
dividend             0
split_coefficient    0
dtype: int64

Rows per ticker:
symbol
AAPL     6539
AMZN     6539
AVGO     4127
GOOG     2960
GOOGL    5377
META     3425
MSFT     6539
NVDA     6539
TSLA     3902
WMT      6539
dtype: int64


## 2. Clean Income Statement

In [23]:
print(f"Shape: {income.shape}")
print(f"\nColumns: {income.columns.tolist()}")
print(f"\nMissing values:\n{income.isnull().sum()}")

Shape: (781, 27)

Columns: ['symbol', 'fiscalDateEnding', 'reportedCurrency', 'grossProfit', 'totalRevenue', 'costOfRevenue', 'costofGoodsAndServicesSold', 'operatingIncome', 'sellingGeneralAndAdministrative', 'researchAndDevelopment', 'operatingExpenses', 'investmentIncomeNet', 'netInterestIncome', 'interestIncome', 'interestExpense', 'nonInterestIncome', 'otherNonOperatingIncome', 'depreciation', 'depreciationAndAmortization', 'incomeBeforeTax', 'incomeTaxExpense', 'interestAndDebtExpense', 'netIncomeFromContinuingOperations', 'comprehensiveIncomeNetOfTax', 'ebit', 'ebitda', 'netIncome']

Missing values:
symbol                                 0
fiscalDateEnding                       0
reportedCurrency                       0
grossProfit                            4
totalRevenue                           0
costOfRevenue                          0
costofGoodsAndServicesSold             0
operatingIncome                        4
sellingGeneralAndAdministrative        2
researchAndDevelo

In [24]:
def clean_fundamentals(df, date_col='fiscalDateEnding'):
    df = df.copy()
    
    # Standardize column names
    df.columns = df.columns.str.strip()
    
    # Standardize symbol column
    if 'Symbol' in df.columns:
        df = df.rename(columns={'Symbol': 'symbol'})
    
    # Convert date
    df[date_col] = pd.to_datetime(df[date_col])
    
    # Convert numeric columns (exclude symbol, date, currency)
    exclude_cols = ['symbol', date_col, 'reportedCurrency']
    numeric_cols = [col for col in df.columns if col not in exclude_cols]
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['symbol', date_col])
    
    # Sort
    df = df.sort_values(['symbol', date_col]).reset_index(drop=True)
    
    return df

income_clean = clean_fundamentals(income)
print(f"Cleaned shape: {income_clean.shape}")
print(f"Date range: {income_clean['fiscalDateEnding'].min()} to {income_clean['fiscalDateEnding'].max()}")

Cleaned shape: (781, 27)
Date range: 2005-12-31 00:00:00 to 2026-01-31 00:00:00


## 3. Clean Balance Sheet

In [25]:
print(f"Shape: {balance.shape}")
print(f"\nMissing values (top 10):")
print(balance.isnull().sum().sort_values(ascending=False).head(10))

Shape: (773, 39)

Missing values (top 10):
otherNonCurrentAssets                     773
investments                               773
longTermDebtNoncurrent                    773
currentDebt                               773
deferredRevenue                           773
accumulatedDepreciationAmortizationPPE    730
currentLongTermDebt                       485
longTermInvestments                       464
treasuryStock                             449
capitalLeaseObligations                   426
dtype: int64


In [26]:
balance_clean = clean_fundamentals(balance)
print(f"Cleaned shape: {balance_clean.shape}")
print(f"Date range: {balance_clean['fiscalDateEnding'].min()} to {balance_clean['fiscalDateEnding'].max()}")

Cleaned shape: (773, 39)
Date range: 2005-12-31 00:00:00 to 2026-01-31 00:00:00


## 4. Clean Cash Flow

In [27]:
print(f"Shape: {cashflow.shape}")
print(f"\nMissing values (top 10):")
print(cashflow.isnull().sum().sort_values(ascending=False).head(10))

Shape: (778, 31)

Missing values (top 10):
proceedsFromRepaymentsOfShortTermDebt                        778
proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet    778
paymentsForRepurchaseOfEquity                                778
paymentsForRepurchaseOfCommonStock                           778
dividendPayoutPreferredStock                                 778
proceedsFromSaleOfTreasuryStock                              778
profitLoss                                                   778
proceedsFromIssuanceOfCommonStock                            778
paymentsForRepurchaseOfPreferredStock                        778
changeInOperatingAssets                                      778
dtype: int64


In [28]:
cashflow_clean = clean_fundamentals(cashflow)
print(f"Cleaned shape: {cashflow_clean.shape}")
print(f"Date range: {cashflow_clean['fiscalDateEnding'].min()} to {cashflow_clean['fiscalDateEnding'].max()}")

Cleaned shape: (778, 31)
Date range: 2005-12-31 00:00:00 to 2026-01-31 00:00:00


## 5. Clean Options

In [29]:
print(f"Shape: {options.shape}")
print(f"\nColumns: {options.columns.tolist()}")
print(f"\nMissing values:\n{options.isnull().sum()}")
print(f"\nSample:")
options.head()

Shape: (3192217, 20)

Columns: ['contractID', 'symbol', 'expiration', 'strike', 'call_put', 'last', 'mark', 'bid', 'bid_size', 'ask', 'ask_size', 'volume', 'open_interest', 'trade_date', 'implied_vol', 'delta', 'gamma', 'theta', 'vega', 'rho']

Missing values:
contractID       0
symbol           0
expiration       0
strike           0
call_put         0
last             0
mark             0
bid              0
bid_size         0
ask              0
ask_size         0
volume           0
open_interest    0
trade_date       0
implied_vol      0
delta            0
gamma            0
theta            0
vega             0
rho              0
dtype: int64

Sample:


,contractID,symbol,expiration,strike,call_put,last,mark,bid,bid_size,ask,ask_size,volume,open_interest,trade_date,implied_vol,delta,gamma,theta,vega,rho
0,AVGO090919C00015000,AVGO,2009-09-19,15.0,call,0.00,0.01,2.50,0,3.30,0,0,0,2009-09-01,0.01488,1.00000,0.00000,-0.00006,0.00000,0.00740
1,AVGO090919P00015000,AVGO,2009-09-19,15.0,put,0.00,0.01,0.00,0,0.25,0,0,0,2009-09-01,0.37584,-0.01672,0.02790,-0.00172,0.00165,-0.00015
2,AVGO090919C00017500,AVGO,2009-09-19,17.5,call,1.20,0.01,0.80,0,1.10,0,0,20,2009-09-01,0.01488,1.00000,0.00000,-0.00007,0.00000,0.00863
3,AVGO090919P00017500,AVGO,2009-09-19,17.5,put,0.40,0.01,0.30,0,0.75,0,0,10,2009-09-01,0.06366,-0.07881,0.58251,-0.00102,0.00583,-0.00070
4,AVGO090919C00020000,AVGO,2009-09-19,20.0,call,0.25,0.01,0.15,0,0.30,0,0,8,2009-09-01,0.25877,0.02565,0.05821,-0.00170,0.00237,0.00022


In [30]:
def clean_options(df):
    df = df.copy()
    
    # Standardize column names
    df.columns = df.columns.str.lower().str.strip()
    
    # Convert dates
    df['expiration'] = pd.to_datetime(df['expiration'])
    
    # Check for trade_date column
    if 'trade_date' in df.columns:
        df['trade_date'] = pd.to_datetime(df['trade_date'])
    
    # Ensure numeric columns
    numeric_cols = ['strike', 'last', 'mark', 'bid', 'ask', 'bid_size', 'ask_size', 
                    'volume', 'open_interest', 'implied_volatility', 
                    'delta', 'gamma', 'theta', 'vega', 'rho']
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Standardize call_put column
    if 'call_put' in df.columns:
        df['call_put'] = df['call_put'].str.upper().str.strip()
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['contractid']) if 'contractid' in df.columns else df.drop_duplicates()
    
    # Sort
    sort_cols = ['symbol', 'trade_date', 'expiration', 'strike'] if 'trade_date' in df.columns else ['symbol', 'expiration', 'strike']
    df = df.sort_values(sort_cols).reset_index(drop=True)
    
    return df

options_clean = clean_options(options)
print(f"Cleaned shape: {options_clean.shape}")
print(f"Expiration range: {options_clean['expiration'].min()} to {options_clean['expiration'].max()}")
print(f"Call/Put split:\n{options_clean['call_put'].value_counts()}")

Cleaned shape: (1121934, 20)
Expiration range: 2008-02-16 00:00:00 to 2028-06-16 00:00:00
Call/Put split:
call_put
CALL    560976
PUT     560958
Name: count, dtype: int64


## 6. Clean Overview

In [31]:
print(f"Shape: {overview.shape}")
print(f"\nColumns: {overview.columns.tolist()}")
overview

Shape: (10, 55)

Columns: ['Symbol', 'AssetType', 'Name', 'Description', 'CIK', 'Exchange', 'Currency', 'Country', 'Sector', 'Industry', 'Address', 'OfficialSite', 'FiscalYearEnd', 'LatestQuarter', 'MarketCapitalization', 'EBITDA', 'PERatio', 'PEGRatio', 'BookValue', 'DividendPerShare', 'DividendYield', 'EPS', 'RevenuePerShareTTM', 'ProfitMargin', 'OperatingMarginTTM', 'ReturnOnAssetsTTM', 'ReturnOnEquityTTM', 'RevenueTTM', 'GrossProfitTTM', 'DilutedEPSTTM', 'QuarterlyEarningsGrowthYOY', 'QuarterlyRevenueGrowthYOY', 'AnalystTargetPrice', 'AnalystRatingStrongBuy', 'AnalystRatingBuy', 'AnalystRatingHold', 'AnalystRatingSell', 'AnalystRatingStrongSell', 'TrailingPE', 'ForwardPE', 'PriceToSalesRatioTTM', 'PriceToBookRatio', 'EVToRevenue', 'EVToEBITDA', 'Beta', '52WeekHigh', '52WeekLow', '50DayMovingAverage', '200DayMovingAverage', 'SharesOutstanding', 'SharesFloat', 'PercentInsiders', 'PercentInstitutions', 'DividendDate', 'ExDividendDate']


,Symbol,AssetType,Name,Description,CIK,Exchange,Currency,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,...,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,AVGO,Common Stock,Broadcom Inc,"Broadcom Inc. is an American designer, develop...",1730168,NASDAQ,USD,USA,TECHNOLOGY,SEMICONDUCTORS,"3421 HILLVIEW AVE, PALO ALTO, CA, UNITED STATE...",https://www.broadcom.com,October,2026-01-31,1516448907000,37218001000,62.47,0.689,59.22,2.48,0.0078,5.12,14.46,0.3660,0.3180,...,1.881,0.164,472.01,8,40,2,0,0,62.47,28.49,22.210,18.73,22.670,41.46,1.257,413.82,137.28,331.56,324.55,4734668000,405875000,1.973,79.523,2026-03-31,2026-03-23
1,GOOGL,Common Stock,Alphabet Inc Class A,Alphabet Inc. is an American multinational con...,1652044,NASDAQ,USD,USA,COMMUNICATION SERVICES,INTERNET CONTENT & INFORMATION,"1600 AMPHITHEATRE PARKWAY, MOUNTAIN VIEW, CA, ...",https://abc.xyz,December,2025-12-31,3715351708000,150175007000,28.41,2.311,34.35,0.83,0.0027,10.81,33.25,0.3280,0.3160,...,0.311,0.180,376.75,11,49,7,0,0,28.41,26.88,9.220,8.96,9.050,20.18,1.112,348.75,140.04,318.54,258.51,5822000000,10833468000,0.575,80.847,2026-03-16,2026-03-09
2,WMT,Common Stock,Walmart Inc. Common Stock,Walmart Inc. is an American multinational reta...,104169,NASDAQ,USD,USA,CONSUMER DEFENSIVE,DISCOUNT STORES,"1 CUSTOMER DRIVE, BENTONVILLE, AR, UNITED STAT...",https://www.stock.walmart.com,January,2026-01-31,965378245000,44027998000,44.52,4.536,12.50,0.94,0.0077,2.72,89.33,0.0307,0.0457,...,-0.190,0.056,136.02,9,30,3,1,0,44.52,40.98,1.354,9.69,1.433,21.99,0.657,134.41,79.11,123.30,107.86,7972403000,4371906000,45.008,38.856,2027-01-04,2026-03-20
3,TSLA,Common Stock,Tesla Inc,"Tesla, Inc. is an American electric vehicle an...",1318605,NASDAQ,USD,USA,CONSUMER CYCLICAL,AUTO MANUFACTURERS,"1 TESLA ROAD, AUSTIN, TX, UNITED STATES, 78725",https://www.tesla.com,December,2025-12-31,1427049808000,10503000000,358.77,5.360,21.90,NaN,NaN,1.06,29.40,0.0400,0.0470,...,-0.606,-0.031,421.61,4,16,17,6,2,358.77,188.68,15.050,17.94,15.230,122.79,1.926,498.83,214.25,417.61,394.08,3752432000,2812710000,11.130,44.595,NaN,NaN
4,AAPL,Common Stock,Apple Inc,Apple Inc. is an American multinational techno...,320193,NASDAQ,USD,USA,TECHNOLOGY,CONSUMER ELECTRONICS,"ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STAT...",https://www.apple.com,September,2025-12-31,3659195744000,152901992000,31.55,2.212,6.00,1.03,0.0041,7.89,29.30,0.2700,0.3540,...,0.183,0.157,295.44,5,25,16,1,1,31.55,28.90,8.400,41.44,8.440,24.06,1.116,288.35,168.48,261.64,246.36,14681140000,14656182000,1.638,65.265,2026-02-12,2026-02-09
5,MSFT,Common Stock,Microsoft Corporation,Microsoft Corporation is an American multinati...,789019,NASDAQ,USD,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,"ONE MICROSOFT WAY, REDMOND, WA, UNITED STATES,...",https://www.microsoft.com,June,2025-12-31,2891343462000,175258993000,24.33,1.275,52.62,3.48,0.0089,15.99,41.10,0.3900,0.4710,...,0.598,0.167,594.62,10,44,3,0,0,24.33,20.45,9.470,7.39,9.350,15.17,1.108,552.24,342.17,424.59,482.33,7425629000,7414788000,0.079,75.974,2026-06-11,2026-05-21
6,NVDA,Common Stock,NVIDIA Corporation,Nvidia Corporation is an American multinationa...,1045810,NASDAQ,USD,USA,TECHNOLOGY,SEMICONDUCTORS,"2788 SAN TOMAS EXPRESSWAY, SANTA CLARA, CA, UN...",https://www.nvidia.com,January,2026-01-31,4339900875000,133230002000,36.52,0.735,6.47,0.04,0.0002,4.89,8.87,0.5560,0.6500,...,0.956,0.732,268.43,11,48,2,1,0,36.52,22.17,20.100,27.59,19.860,29.66,2.375,212.17,86.60,185.10,17

In [32]:
def clean_overview(df):
    df = df.copy()
    
    # Standardize symbol column
    df = df.rename(columns={'Symbol': 'symbol'})
    
    # Select relevant columns for our project
    keep_cols = [
        'symbol', 'Name', 'Sector', 'Industry', 'Exchange',
        'SharesOutstanding', 'DividendYield', 'Beta'
    ]
    
    # Keep only columns that exist
    keep_cols = [col for col in keep_cols if col in df.columns]
    df = df[keep_cols]
    
    # Standardize column names
    df.columns = df.columns.str.lower()
    
    # Convert numeric columns
    numeric_cols = ['sharesoutstanding', 'dividendyield', 'beta']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

overview_clean = clean_overview(overview)
print(f"Cleaned shape: {overview_clean.shape}")
overview_clean

Cleaned shape: (10, 8)


,symbol,name,sector,industry,exchange,sharesoutstanding,dividendyield,beta
0,AVGO,Broadcom Inc,TECHNOLOGY,SEMICONDUCTORS,NASDAQ,4734668000,0.0078,1.257
1,GOOGL,Alphabet Inc Class A,COMMUNICATION SERVICES,INTERNET CONTENT & INFORMATION,NASDAQ,5822000000,0.0027,1.112
2,WMT,Walmart Inc. Common Stock,CONSUMER DEFENSIVE,DISCOUNT STORES,NASDAQ,7972403000,0.0077,0.657
3,TSLA,Tesla Inc,CONSUMER CYCLICAL,AUTO MANUFACTURERS,NASDAQ,3752432000,NaN,1.926
4,AAPL,Apple Inc,TECHNOLOGY,CONSUMER ELECTRONICS,NASDAQ,14681140000,0.0041,1.116
5,MSFT,Microsoft Corporation,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,NASDAQ,7425629000,0.0089,1.108
6,NVDA,NVIDIA Corporation,TECHNOLOGY,SEMICONDUCTORS,NASDAQ,24300000000,0.0002,2.375
7,AMZN,Amazon.com Inc,CONSUMER CYCLICAL,INTERNET RETAIL,NASDAQ,10734921000,NaN,1.420
8,GOOG,Alphabet Inc Class C,COMMUNICATION SERVICES,INTERNET CONTENT & INFORMATION,NASDAQ,5438000000,0.0027,1.112
9,META,Meta Platforms Inc.,COMMUNICATION SERVICES,INTERNET CONTENT & INFORMATION,NASDAQ,2187178000,0.0034,1.279


## 7. Data Quality Summary

In [33]:
datasets = {
    'daily': daily_clean,
    'income': income_clean,
    'balance': balance_clean,
    'cashflow': cashflow_clean,
    'options': options_clean,
    'overview': overview_clean
}

print("----- Data Quality Summary -----")
for name, df in datasets.items():
    missing_pct = (df.isnull().sum().sum() / df.size) * 100
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Missing: {missing_pct:.2f}%")
    print(f"  Tickers: {df['symbol'].nunique()}")

----- Data Quality Summary -----

daily:
  Shape: (52486, 10)
  Missing: 0.00%
  Tickers: 10

income:
  Shape: (781, 27)
  Missing: 25.91%
  Tickers: 10

balance:
  Shape: (773, 39)
  Missing: 25.41%
  Tickers: 10

cashflow:
  Shape: (778, 31)
  Missing: 53.99%
  Tickers: 10

options:
  Shape: (1121934, 20)
  Missing: 0.00%
  Tickers: 10

overview:
  Shape: (10, 8)
  Missing: 2.50%
  Tickers: 10


## 8. Save Cleaned Data

In [34]:
# Save to processed folder
daily_clean.to_parquet(PROCESSED / 'daily_clean.parquet', index=False)
income_clean.to_parquet(PROCESSED / 'income_clean.parquet', index=False)
balance_clean.to_parquet(PROCESSED / 'balance_clean.parquet', index=False)
cashflow_clean.to_parquet(PROCESSED / 'cashflow_clean.parquet', index=False)
options_clean.to_parquet(PROCESSED / 'options_clean.parquet', index=False)
overview_clean.to_parquet(PROCESSED / 'overview_clean.parquet', index=False)

print("Cleaned data saved to:")
for f in PROCESSED.glob('*.parquet'):
    print(f"  {f}")

Cleaned data saved to:
  ../data/processed/options_clean.parquet
  ../data/processed/cashflow_clean.parquet
  ../data/processed/modeling_data.parquet
  ../data/processed/balance_clean.parquet
  ../data/processed/overview_clean.parquet
  ../data/processed/daily_clean.parquet
  ../data/processed/income_clean.parquet
  ../data/processed/features.parquet
  ../data/processed/daily_with_returns.parquet
